In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import gc

<h2>Early Concatting, Transforming, and Merging</h2>

<h3>Concat Raw Datasets</h3>

In [31]:
# Concat all facility csv files into a single df

facility_folder = Path('Facility')

df_list = []
for file in facility_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_facility = pd.concat(df_list, ignore_index=True)

output_path = Path('Facility/Concatted File/facility_2019_2024.csv')
campd_facility.to_csv(output_path, index=False)

In [2]:
# Concat all MT csv files into a single df

mt_folder = Path('MT/Hourly')

df_list = []
for file in mt_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_mt = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs as one

campd_mt.to_csv('MT/Hourly/Concatted File/mt_2019_2024.csv', index=False)

del campd_mt
gc.collect()

0

In [17]:
# get all EIA data

eia_folder = Path('EIA')

df_list = []
for file in eia_folder.iterdir():
    if file.is_file() and file.name.endswith('.xlsx'):
        # extract the year from filename and add as column
        match = re.search(r'_Y(\d{4})\.xlsx', str(file))
        if match:
            year = int(match.group(1))
            df = pd.read_excel(file, sheet_name="Operable", header=1)
            df["Year"] = year
            df_list.append(df)

eia_generator = pd.concat(df_list, ignore_index=True)

# filter for rows in campd_ny, campd_wy, and campd_tx
eia_generator = eia_generator[eia_generator["Plant Code"].isin(campd_mt["Facility ID"])]

output_path = Path('EIA/Concatted File/eia_generator_2019_2023_MT.csv')
eia_generator.to_csv(output_path, index=False)

<h3>Transform Emission Data</h3>

In [3]:
# define data type dictionary for campd dataframes
campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Associated Stacks": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "PM Controls": "string",
    "Hg Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Operating Time": "float64"
}

<h4>Montana</h4>

In [4]:
campd_mt = pd.read_csv('MT/Hourly/Concatted File/mt_2019_2024.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [5]:
campd_mt.shape

(620280, 32)

<h3>Transform Facility Data</h3>

In [6]:
# read facility data and assign data types

campd_facility = pd.read_csv('Facility/Concatted File/facility_2019_2024.csv')

# limit to columns we need
campd_facility = campd_facility[["Year", "State", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)']]
campd_facility = campd_facility.astype({
    'Year': 'Int64',
    'Facility ID': 'Int64',
    'Unit ID': 'string',
    "State": "string",
    'EPA Region': 'Int64',
    "Source Category": 'string',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'SO2 Phase': 'string',
    'NOx Phase': 'string',
    'Commercial Operation Date': 'string',
    'Operating Status': 'string',
    'Max Hourly HI Rate (mmBtu/hr)': 'float64',
    'Associated Generators & Nameplate Capacity (MWe)': 'string'
    })

# filter out rows that are not in campd_mt
campd_facility = campd_facility[
    campd_facility["Facility ID"].isin(campd_mt["Facility ID"])
]

In [7]:
# function to extract and sum nameplate capacities in facility data
def extract_amt_sum_nameplate_capacity(data):
    data = str(data).strip()  # Convert data to string and remove leading/trailing whitespace
    matches = re.findall(r'\((\d+(?:\.\d+)?)\)', data) # find floats or integers in parentheses (can be float like 191.8 or integer like 170)

    if matches:
        total_sum = 0.0
        for match in matches:
            total_sum += float(match)
        #print("Total sum: ", total_sum)
        return total_sum
    else:
        #print("No matches found")
        return None

campd_facility['Nameplate Capacity (MWe)'] = campd_facility['Associated Generators & Nameplate Capacity (MWe)'].apply(extract_amt_sum_nameplate_capacity)
campd_facility.to_csv('Facility/Concatted File/facility_2019_2024_MT.csv', index=False)

<h3>Merge Facility and Emission dfs</h3>

(and do early data cleaning based on this merge)

<h4>Montana</h4>

In [8]:
# Look for mismatched Unit IDs between campd_facility and campd_mt

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "MT"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_mt[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs


In [10]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_mt["Unit ID"] = campd_mt["Unit ID"].replace("001", "1")
campd_mt["Unit ID"] = campd_mt["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [12]:
# check after transformation

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "MT"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_mt[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)

# the 001 -> 1 transformation fixed none here

,Facility ID,Mismatched Unit Count,Mismatching Unit IDs


In [13]:
campd_facility_mt = campd_facility[campd_facility["State"] == "MT"]
campd_facility_mt = campd_facility_mt[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_ny for the join
campd_mt['Year'] = campd_mt['Date'].dt.year
campd_mt_facilities = pd.merge(campd_mt, campd_facility_mt, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [14]:
check = campd_mt_facilities.isna().sum()
check[check > 0]

Associated Stacks               304632
Gross Load (MW)                 306447
Steam Load (1000 lb/hr)         620280
SO2 Mass (lbs)                  306447
SO2 Mass Measure Indicator      306447
SO2 Rate (lbs/mmBtu)            306447
SO2 Rate Measure Indicator      306447
CO2 Mass (short tons)           306447
CO2 Mass Measure Indicator      306447
CO2 Rate (short tons/mmBtu)     306447
CO2 Rate Measure Indicator      306447
NOx Mass (lbs)                  306450
NOx Mass Measure Indicator      306450
NOx Rate (lbs/mmBtu)            306450
NOx Rate Measure Indicator      306450
Heat Input (mmBtu)              306447
Heat Input Measure Indicator    306447
Secondary Fuel Type             232320
SO2 Controls                    420864
PM Controls                     420864
Hg Controls                     473472
NOx Phase                       473472
dtype: int64

In [15]:
campd_mt_facilities.to_csv('MT/Hourly/Concatted File/mt_2019_2024_with_facility.csv', index=False)

<h3>Transform EIA-860 Data</h3>

(has additional data about generator nameplate capacities and age, but needs to be aggregated to a unit level)

In [18]:
# read eia generator data
eia_generator = pd.read_csv('EIA/Concatted File/eia_generator_2019_2023_MT.csv')

# these cols have blank strings (" ") that should be nan
columns_to_clean = [
    "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)",
    "Minimum Load (MW)", "Planned Retirement Year"
]

# replace blank strings with nan (any strings with only whitespace)
eia_generator[columns_to_clean] = eia_generator[columns_to_clean].replace(r'^\s*$', np.nan, regex=True)

# now define full dtype dictionary
eia_generator_type_dict = {
    "Year": "Int64", 
    "Utility ID": "string",
    "Utility Name": "string",
    "Plant Code": "Int64",  
    "Plant Name": "string",
    "State": "string",
    "Generator ID": "string",
    "Technology": "string",
    "Prime Mover": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Power Factor": "float64", 
    "Summer Capacity (MW)": "float64",  
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Operating Year": "Int64",
    "Planned Retirement Year": "float64"
}

# filter needed cols
eia_generator = eia_generator[["Year","Utility ID","Utility Name", "Plant Code", "Plant Name", "State", "Generator ID", "Technology", "Prime Mover", "Nameplate Capacity (MW)", "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)", "Minimum Load (MW)", "Operating Year", "Planned Retirement Year"]]

# cast all dtypes according to the dictionary
eia_generator = eia_generator.astype(eia_generator_type_dict)


<h4>old manual version without crosswalk</h4>

In [19]:
# Transform and aggregate to a unit-level (using facility data to add unit IDs)

# extract Generator IDs from the associated string (in campd_facility) and put them in a list 
def extract_generator_ids(data):
    data = str(data).strip()
    # format should be e. g. "G1 (170), G2 (191.8)"
    matches = re.findall(r'([A-Za-z0-9]+)\s*\(\d+(?:\.\d+)?\)', data)
    return matches if matches else []

campd_facility["Generator ID List"] = campd_facility["Associated Generators & Nameplate Capacity (MWe)"].apply(extract_generator_ids)

# explode so each row is Unit ID + one Generator ID -> create a year, facility, unit, generator mapping (a unit may have multiple generators)
unit_generator_map = campd_facility[["Year", "Facility ID", "Unit ID", "Generator ID List",]].explode("Generator ID List")
unit_generator_map = unit_generator_map.rename(columns={"Generator ID List": "Generator ID"})
display(unit_generator_map)

# merge with eia generator-level dataset
# (first rename Plant Code in eia_generator to Facility ID to match unit_generator_map)
eia_generator_renamed = eia_generator.rename(columns={"Plant Code": "Facility ID"})
# merge on Facility ID and Generator ID
joined = unit_generator_map.merge(eia_generator_renamed, on=["Year","Facility ID", "Generator ID"], how="inner")
display(joined)

,Year,Facility ID,Unit ID,Generator ID
817,2019,2176,GT-2,2
1725,2019,6076,1,1
1726,2019,6076,2,2
1727,2019,6076,3,3
1728,2019,6076,4,4
...,...,...,...,...
25089,2024,56908,1B,1
25090,2024,56908,2A,2
25091,2024,56908,2B,2
25092,2024,56908,3A,3


,Year,Facility ID,Unit ID,Generator ID,Utility ID,Utility Name,Plant Name,State,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,2019,6076,1,1,15298,Talen Montana LLC,Colstrip,MT,Conventional Steam Coal,ST,358.0,0.95,307.0,307.0,90.0,1975,2020.0
1,2019,6076,2,2,15298,Talen Montana LLC,Colstrip,MT,Conventional Steam Coal,ST,358.0,0.95,307.0,307.0,90.0,1976,2020.0
2,2019,6076,3,3,15298,Talen Montana LLC,Colstrip,MT,Conventional Steam Coal,ST,823.7,0.95,740.0,740.0,200.0,1984,NaN
3,2019,6076,4,4,15298,Talen Montana LLC,Colstrip,MT,Conventional Steam Coal,ST,823.7,0.95,740.0,740.0,200.0,1986,NaN
4,2019,6089,B1,1,12199,Montana-Dakota Utilities Co,Lewis & Clark,MT,Conventional Steam Coal,ST,50.0,0.98,53.1,44.0,19.0,1958,2021.0
5,2019,55749,U1,UNT1,16233,Heorot Power Management,Hardin Generator Project,MT,Conventional Steam Coal,ST,115.7,0.85,107.0,107.0,70.0,2006,NaN
6,2019,56606,CT01,01,1307,Basin Electric Power Coop,Culbertson Generation Station,MT,Natural Gas Fired Combustion Turbine,GT,91.0,0.85,85.0,95.0,46.1,2010,NaN
7,2019,56908,1A,1,56815,NorthWestern Energy DGGS,Dave Gates Generating Station,MT,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN
8,2019,56908,1B,1,56815,NorthWestern Energy DGGS,Dave Gates Generating Station,MT,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN
9,2019,56908,2A,2,56815,NorthWestern Energy DGGS,Dave Gates Generating Station,MT,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN


In [20]:
# aggregate the resulting generator-level df to a unit-level df

agg_funcs = {
    'Nameplate Capacity (MW)': 'sum',
    'Summer Capacity (MW)': 'sum',
    'Winter Capacity (MW)': 'sum',
    'Minimum Load (MW)': 'sum',
    'Operating Year': ['min', 'max', 'mean'], # get operating year of the oldest (min) and newest (max) generator in the unit as well as the average across all gens
    'Planned Retirement Year': 'max',
}

unit_level = joined.groupby(['Year', 'Facility ID', 'Unit ID']).agg(agg_funcs).reset_index()

display(unit_level)

# flatten multi-index columns (only keep the aggregation operator (min, max, mean) for Operating Year)
unit_level.columns = [
    f"{col[0]}_{col[1]}" if col[0] == "Operating Year" else col[0]
    for col in unit_level.columns.to_flat_index()
]

unit_level

Year Facility ID Unit ID Nameplate Capacity (MW) Summer Capacity (MW)  \
                                                 sum                  sum   
0   2019        6076       1                   358.0                307.0   
1   2019        6076       2                   358.0                307.0   
2   2019        6076       3                   823.7                740.0   
3   2019        6076       4                   823.7                740.0   
4   2019        6089      B1                    50.0                 53.1   
5   2019       55749      U1                   115.7                107.0   
6   2019       56606    CT01                    91.0                 85.0   
7   2019       56908      1A                    67.7                 44.1   
8   2019       56908      1B                    67.7                 44.1   
9   2019       56908      2A                    67.7                 44.1   
10  2019       56908      2B                    67.7                 44.1   
11  2019       56908      3A                    67.7                 44.1   
12  2019       56908      3B                    67.7                 44.1   
13  2020        6076       3                   823.7                740.0   
14  2020        6076       4                   823.7                740.0   
15  2020        6089      B1                    50.0                 53.1   
16  2020       55749      U1                   115.7                107.0   
17  2020       56606    CT01                    91.0                 85.0   
18  2020       56908      1A                    67.7                 44.1   
19  2020       56908      1B                    67.7                 44.1   
20  2020       56908      2A                    67.7                 44.1   
21  2020       56908      2B                    67.7                 44.1   
22  2020       56908      3A                    67.7                 44.1   
23  2020       56908      3B                    67.7                 44.1   
24  2021        6076       3                   823.7                740.0   
25  2021        6076       4                   823.7                740.0   
26  2021       55749      U1                   115.7                107.0   
27  2021       56606    CT01                   108.2                 85.0   
28  2021       56908      1A                    67.7                 44.1   
29  2021       56908      1B                    67.7                 44.1   
30  2021       56908      2A                    67.7                 44.1   
31  2021       56908      2B                    67.7                 44.1   
32  2021       56908      3A                    67.7                 44.1   
33  2021       56908      3B                    67.7                 44.1   
34  2022        6076       3                   823.7                740.0   
35  2022        6076       4                   823.7                740.0   
36  2022       55749      U1                   115.7                107.0   
37  2022       56606    CT01                   108.2                 85.0   
38  2022       56908      1A                    67.7                 44.1   
39  2022       56908      1B                    67.7                 44.1   
40  2022       56908      2A                    67.7                 44.1   
41  2022       56908      2B                    67.7                 44.1   
42  2022       56908      3A                    67.7                 44.1   
43  2022       56908      3B                    67.7                 44.1   
44  2023        6076       3                   823.7                740.0   
45  2023        6076       4                   823.7                740.0   
46  2023       55749      U1                   115.7                107.0   
47  2023       56606    CT01                   108.2                 85.0   
48  2023       56908      1A                    67.7                 44.1   
49  2023       56908      1B                    67.7                 44.1   


,Year,Facility ID,Unit ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year
0,2019,6076,1,358.0,307.0,307.0,90.0,1975,1975,1975.0,2020.0
1,2019,6076,2,358.0,307.0,307.0,90.0,1976,1976,1976.0,2020.0
2,2019,6076,3,823.7,740.0,740.0,200.0,1984,1984,1984.0,NaN
3,2019,6076,4,823.7,740.0,740.0,200.0,1986,1986,1986.0,NaN
4,2019,6089,B1,50.0,53.1,44.0,19.0,1958,1958,1958.0,2021.0
5,2019,55749,U1,115.7,107.0,107.0,70.0,2006,2006,2006.0,NaN
6,2019,56606,CT01,91.0,85.0,95.0,46.1,2010,2010,2010.0,NaN
7,2019,56908,1A,67.7,44.1,48.1,5.0,2011,2011,2011.0,NaN
8,2019,56908,1B,67.7,44.1,48.1,5.0,2011,2011,2011.0,NaN
9,2019,56908,2A,67.7,44.1,48.1,5.0,2011,2011,2011.0,NaN


In [21]:
# set data types again for the final unit_level dataframe
unit_level = unit_level.astype({
    'Year': 'Int64',
    'Facility ID': 'Int64',
    'Unit ID': 'string',
    'Nameplate Capacity (MW)': 'float64',
    'Summer Capacity (MW)': 'float64',
    'Winter Capacity (MW)': 'float64',
    'Minimum Load (MW)': 'float64',
    'Operating Year_min': 'Int64',
    'Operating Year_max': 'Int64',
    'Operating Year_mean': 'Int64',
    'Planned Retirement Year': 'Int64'
})

In [22]:
# Artificially extend the unit_level dataframe to include 2024 data (just copy the 2023 data and set Year to 2024)
unit_level_2023 = unit_level[unit_level["Year"] == 2023].copy()
unit_level_2024 = unit_level_2023.copy()
unit_level_2024["Year"] = 2024
unit_level_extended = pd.concat([unit_level, unit_level_2024], ignore_index=True)

In [23]:
# Calc min, max, and mean ages of the generators in the unit
unit_level_extended['unit_age_min'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_min']
unit_level_extended['unit_age_max'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_max']
unit_level_extended['unit_age_mean'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_mean']

In [24]:
# save to csv
unit_level_extended.to_csv('EIA/Concatted File/eia_unit_level_2019_2024_MT.csv', index=False)

In [25]:
unit_level_extended_manually = pd.read_csv('EIA/Concatted File/eia_unit_level_2019_2024_MT.csv')
unit_level_extended_manually

,Year,Facility ID,Unit ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean
0,2019,6076,1,358.0,307.0,307.0,90.0,1975,1975,1975,2020.0,44,44,44
1,2019,6076,2,358.0,307.0,307.0,90.0,1976,1976,1976,2020.0,43,43,43
2,2019,6076,3,823.7,740.0,740.0,200.0,1984,1984,1984,NaN,35,35,35
3,2019,6076,4,823.7,740.0,740.0,200.0,1986,1986,1986,NaN,33,33,33
4,2019,6089,B1,50.0,53.1,44.0,19.0,1958,1958,1958,2021.0,61,61,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,2024,56908,1B,67.7,44.1,48.1,5.0,2011,2011,2011,NaN,13,13,13
60,2024,56908,2A,67.7,44.1,48.1,5.0,2011,2011,2011,NaN,13,13,13
61,2024,56908,2B,67.7,44.1,48.1,5.0,2011,2011,2011,NaN,13,13,13
62,2024,56908,3A,67.7,44.1,48.1,5.0,2011,2011,2011,NaN,13,13,13


<h3>Merge Emission data with EIA data</h3>

In [26]:
campd_mt_facilities = pd.read_csv('MT/Hourly/Concatted File/mt_2019_2024_with_facility.csv', dtype=campd_type_dict, date_format=["Date"])

C:\Users\marce\AppData\Local\Temp\ipykernel_10284\810564510.py:1: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  campd_mt_facilities = pd.read_csv('MT/Hourly/Concatted File/mt_2019_2024_with_facility.csv', dtype=campd_type_dict, date_format=["Date"])


In [27]:
complete_campd_mt = pd.merge(campd_mt_facilities, unit_level_extended_manually, on=['Year', 'Facility ID', 'Unit ID'], how='left')

In [28]:
# Filter where the values are different (excluding NaNs where both are the same)
mismatch_df = complete_campd_mt[
    complete_campd_mt["Nameplate Capacity (MWe)"] != complete_campd_mt["Nameplate Capacity (MW)"]
][["Year", "Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"]].drop_duplicates()

# Count where either column is NaN
nan_mwe = complete_campd_mt["Nameplate Capacity (MWe)"].isna().sum()
nan_mw = complete_campd_mt["Nameplate Capacity (MW)"].isna().sum()

# Display results
print("Unique mismatching rows (with year):")
display(mismatch_df)

print(f"\nNaN counts:")
print(f"Nameplate Capacity (MWe): {nan_mwe}")
print(f"Nameplate Capacity (MW): {nan_mw}")

Unique mismatching rows (with year):


,Year,Nameplate Capacity (MWe),Nameplate Capacity (MW)
0,2019,43.0,NaN
8760,2019,358.4,358.0
26280,2019,778.0,823.7
52560,2019,121.0,115.7
61320,2019,100.0,91.0
122640,2020,43.0,NaN
131424,2020,358.4,NaN
135792,2020,778.0,823.7
162144,2020,121.0,115.7
170928,2020,100.0,91.0



NaN counts:
Nameplate Capacity (MWe): 0
Nameplate Capacity (MW): 59136


In [29]:
# Create a column for absolute difference
complete_campd_mt["Capacity Difference"] = (
    complete_campd_mt["Nameplate Capacity (MWe)"] - complete_campd_mt["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_mt.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference: {average_difference:.2f} MW")

# Sort by largest difference
top_differences = diff_df.sort_values(by="Capacity Difference", ascending=False)

# Display top 10 mismatches
display(top_differences[["Year", "Facility ID", "Unit ID", 
                         "Nameplate Capacity (MWe)", "Nameplate Capacity (MW)", 
                         "Capacity Difference"]].head(10))


Average Capacity Difference: 9.87 MW


,Year,Facility ID,Unit ID,Nameplate Capacity (MWe),Nameplate Capacity (MW),Capacity Difference
151244,2020,6076,4,778.0,823.7,45.7
147367,2020,6076,4,778.0,823.7,45.7
147387,2020,6076,4,778.0,823.7,45.7
147388,2020,6076,4,778.0,823.7,45.7
147389,2020,6076,4,778.0,823.7,45.7
147390,2020,6076,4,778.0,823.7,45.7
147391,2020,6076,4,778.0,823.7,45.7
147392,2020,6076,4,778.0,823.7,45.7
147393,2020,6076,4,778.0,823.7,45.7
147394,2020,6076,4,778.0,823.7,45.7


In [30]:
# fill EIA columns with na if it capacity difference exceeds a certain threshold

# rename the EIA column to make it clearer
for df in [complete_campd_mt]:
    df.rename(columns={
        "Nameplate Capacity (MW)": "Nameplate Capacity (MW) - EIA"
    }, inplace=True)

threshold = 30 
columns_to_null = [
    "Nameplate Capacity (MW) - EIA", 
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)"
]

# Recalculate capacity difference
complete_campd_mt["Capacity Difference"] = (
    complete_campd_mt["Nameplate Capacity (MWe)"] - complete_campd_mt["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_mt["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_mt.loc[condition, columns_to_null] = np.nan

In [31]:
navals = complete_campd_mt[["Nameplate Capacity (MW) - EIA", "Nameplate Capacity (MWe)"]].isna().sum()
navals_percent = (navals / len(complete_campd_mt)) * 100
navals_percent.sort_values(ascending=False)

Nameplate Capacity (MW) - EIA    26.496421
Nameplate Capacity (MWe)          0.000000
dtype: float64

In [32]:
complete_campd_mt.to_csv('MT/Hourly/Concatted File/mt_2019_2024_with_manualEIA.csv', index=False)

<h2>Data Cleaning of Final Datasets</h2>

In [33]:
# relevant columns
complete_campd_columns = [
    "State",
    "Facility Name",
    "Facility ID",
    "Unit ID",
    "Gross Load (MW)",
    "Steam Load (1000 lb/hr)",
    "SO2 Mass (lbs)",
    "SO2 Mass Measure Indicator",
    "SO2 Rate (lbs/mmBtu)",
    "SO2 Rate Measure Indicator",
    "CO2 Mass (short tons)",
    "CO2 Mass Measure Indicator",
    "CO2 Rate (short tons/mmBtu)",
    "CO2 Rate Measure Indicator",
    "NOx Mass (lbs)",
    "NOx Mass Measure Indicator",
    "NOx Rate (lbs/mmBtu)",
    "NOx Rate Measure Indicator",
    "Heat Input (mmBtu)",
    "Heat Input Measure Indicator",
    "Primary Fuel Type",
    "Secondary Fuel Type",
    "Unit Type",
    "SO2 Controls",
    "NOx Controls",
    "Program Code",
    "Hour",
    "Year",
    "Date",
    "Operating Time",
    "Source Category",
    "Latitude",
    "Longitude",
    "SO2 Phase",
    "NOx Phase",
    "Operating Status",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Nameplate Capacity (MWe)",
    "Nameplate Capacity (MW) - EIA",
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)",
    "Planned Retirement Year",
    "unit_age_min",
    "unit_age_max",
    "unit_age_mean"
]

# Filtered dtype dictionary
complete_campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Year": "Int64",
    "Operating Time": "float64",
    "Source Category": "string",
    "Latitude": "float64",
    "Longitude": "float64",
    "SO2 Phase": "string",
    "NOx Phase": "string",
    "Operating Status": "string",
    "Max Hourly HI Rate (mmBtu/hr)": "float64",
    "Associated Generators & Nameplate Capacity (MWe)": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Capacity (MW) - EIA": "float64",
    "Summer Capacity (MW)": "float64",
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Planned Retirement Year": "Int64",
    "unit_age_min": "Int64",
    "unit_age_max": "Int64",
    "unit_age_mean": "float64"
}


<h3>Montana</h3>

In [34]:
complete_campd_mt = pd.read_csv(
    'MT/Hourly/Concatted File/mt_2019_2024_with_manualEIA.csv',  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract day and month as new col from date
complete_campd_mt["Month"] = complete_campd_mt["Date"].dt.month
complete_campd_mt["Day"] = complete_campd_mt["Date"].dt.day

display(complete_campd_mt.head())
print(complete_campd_mt.dtypes)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Nameplate Capacity (MW) - EIA,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day
0,MT,Glendive Generating Station,2176,GT-2,2019-01-01,0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
1,MT,Glendive Generating Station,2176,GT-2,2019-01-01,1,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
2,MT,Glendive Generating Station,2176,GT-2,2019-01-01,2,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
3,MT,Glendive Generating Station,2176,GT-2,2019-01-01,3,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
4,MT,Glendive Generating Station,2176,GT-2,2019-01-01,4,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1


State                                               string[python]
Facility Name                                       string[python]
Facility ID                                                  Int64
Unit ID                                             string[python]
Date                                                datetime64[ns]
Hour                                                         Int64
Operating Time                                             float64
Gross Load (MW)                                            float64
Steam Load (1000 lb/hr)                                    float64
SO2 Mass (lbs)                                             float64
SO2 Mass Measure Indicator                          string[python]
SO2 Rate (lbs/mmBtu)                                       float64
SO2 Rate Measure Indicator                          string[python]
CO2 Mass (short tons)                                      float64
CO2 Mass Measure Indicator                          string[pyt

In [35]:
print(complete_campd_mt.shape)
null_mt = complete_campd_mt.isnull().sum()
null_mt = null_mt[null_mt > 0].sort_values(ascending=False)
display(null_mt)

# Give null vals as percentage of total rows
total_rows = complete_campd_mt.shape[0]
null_percentages = (null_mt / total_rows) * 100

null_percentages.sort_values(ascending=False)

(620280, 49)


Steam Load (1000 lb/hr)          620280
Planned Retirement Year          585216
NOx Phase                        473472
SO2 Controls                     420864
NOx Rate Measure Indicator       306450
NOx Mass (lbs)                   306450
NOx Rate (lbs/mmBtu)             306450
NOx Mass Measure Indicator       306450
Heat Input Measure Indicator     306447
Heat Input (mmBtu)               306447
Gross Load (MW)                  306447
CO2 Rate Measure Indicator       306447
CO2 Rate (short tons/mmBtu)      306447
CO2 Mass Measure Indicator       306447
CO2 Mass (short tons)            306447
SO2 Rate Measure Indicator       306447
SO2 Rate (lbs/mmBtu)             306447
SO2 Mass Measure Indicator       306447
SO2 Mass (lbs)                   306447
Secondary Fuel Type              232320
Nameplate Capacity (MW) - EIA    164352
Summer Capacity (MW)             164352
Winter Capacity (MW)             164352
Minimum Load (MW)                164352
unit_age_min                      59136


Steam Load (1000 lb/hr)          100.000000
Planned Retirement Year           94.347069
NOx Phase                         76.331979
SO2 Controls                      67.850648
NOx Rate Measure Indicator        49.405107
NOx Mass (lbs)                    49.405107
NOx Rate (lbs/mmBtu)              49.405107
NOx Mass Measure Indicator        49.405107
SO2 Mass (lbs)                    49.404624
SO2 Mass Measure Indicator        49.404624
SO2 Rate (lbs/mmBtu)              49.404624
SO2 Rate Measure Indicator        49.404624
CO2 Mass (short tons)             49.404624
CO2 Mass Measure Indicator        49.404624
CO2 Rate (short tons/mmBtu)       49.404624
CO2 Rate Measure Indicator        49.404624
Gross Load (MW)                   49.404624
Heat Input (mmBtu)                49.404624
Heat Input Measure Indicator      49.404624
Secondary Fuel Type               37.454053
Nameplate Capacity (MW) - EIA     26.496421
Summer Capacity (MW)              26.496421
Winter Capacity (MW)            

In [36]:
# handle categorical null values appropriately

cat_cols_with_na_none = [
    "SO2 Controls",
    "NOx Phase",
    "Secondary Fuel Type",
    "NOx Controls",
    "SO2 Phase"
]

cat_cols_with_na_unknown = [
    "Operating Status",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Source Category",
    "CO2 Rate Measure Indicator",
    "SO2 Rate Measure Indicator",
    "NOx Rate Measure Indicator", "Program Code"
]

# here nan means that no data was likely purposely reported -> fill with "Nothing"
for col in cat_cols_with_na_none:
    complete_campd_mt[col] = complete_campd_mt[col].fillna("Nothing").astype("string")

# here nan means that no data was likely reported -> fill with "Missing"
for col in cat_cols_with_na_unknown:
    complete_campd_mt[col] = complete_campd_mt[col].fillna("Missing").astype("string")

num_cols_with_na = [
    "Planned Retirement Year",
    "unit_age_max",
    "unit_age_min",
    "unit_age_mean",
    "Minimum Load (MW)",
    "Winter Capacity (MW)",
    "Summer Capacity (MW)",
    #"Nameplate Capacity (MW) - EIA", # dropped later
    "Longitude",
    "Latitude",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Nameplate Capacity (MWe)"
]

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_mt = complete_campd_mt.drop(columns=["Nameplate Capacity (MW) - EIA"])

print("Null value percentages in complete_campd_mt after adjustments:")
total_rows = complete_campd_mt.shape[0]
null_mt = complete_campd_mt.isnull().sum()
null_mt = null_mt[null_mt > 0].sort_values(ascending=False)
null_percentages = (null_mt / total_rows) * 100
null_percentages.sort_values(ascending=False)

Null value percentages in complete_campd_mt after adjustments:


Steam Load (1000 lb/hr)         100.000000
Planned Retirement Year          94.347069
NOx Rate (lbs/mmBtu)             49.405107
NOx Mass (lbs)                   49.405107
NOx Mass Measure Indicator       49.405107
CO2 Mass Measure Indicator       49.404624
SO2 Mass (lbs)                   49.404624
SO2 Mass Measure Indicator       49.404624
SO2 Rate (lbs/mmBtu)             49.404624
CO2 Mass (short tons)            49.404624
CO2 Rate (short tons/mmBtu)      49.404624
Gross Load (MW)                  49.404624
Heat Input (mmBtu)               49.404624
Heat Input Measure Indicator     49.404624
Summer Capacity (MW)             26.496421
Winter Capacity (MW)             26.496421
Minimum Load (MW)                26.496421
unit_age_min                      9.533759
unit_age_max                      9.533759
unit_age_mean                     9.533759
dtype: float64

In [37]:
# measurement methods
print("Measurement methods in complete_campd_mt:")
display(complete_campd_mt.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_mt.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_mt.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())

Measurement methods in complete_campd_mt:


CO2 Mass Measure Indicator
Measured                   310530
Measured and Substitute      3159
Substitute                    144
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Calculated    313830
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
Measured                   310632
Measured and Substitute      2257
Substitute                    944
Name: SO2 Mass Measure Indicator, dtype: int64

In [38]:
# calculate hourly capacity factor
complete_campd_mt["hourly_capacity_factor"] = (
    complete_campd_mt["Gross Load (MW)"] * complete_campd_mt["Operating Time"]
) / complete_campd_mt["Nameplate Capacity (MWe)"]

complete_campd_mt.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,MT,Glendive Generating Station,2176,GT-2,2019-01-01,0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN
1,MT,Glendive Generating Station,2176,GT-2,2019-01-01,1,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN
2,MT,Glendive Generating Station,2176,GT-2,2019-01-01,2,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN
3,MT,Glendive Generating Station,2176,GT-2,2019-01-01,3,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN
4,MT,Glendive Generating Station,2176,GT-2,2019-01-01,4,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN


In [39]:
# calculate hourly ramping factor
all_hours_mt = complete_campd_mt
all_hours_mt["Gross Load (MW)"] = all_hours_mt["Gross Load (MW)"].fillna(0)


# Sort data by unit and time to calculate t-1 values
all_hours_mt = all_hours_mt.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_mt["gross_load_lag1"] = all_hours_mt.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_mt["hourly_ramping_factor"] = (
    all_hours_mt["Gross Load (MW)"] - all_hours_mt["gross_load_lag1"]
) / all_hours_mt["Nameplate Capacity (MWe)"]

display(all_hours_mt.head(10))

complete_campd_mt = complete_campd_mt.merge(all_hours_mt[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                        on=["Facility ID", "Unit ID", "Date", "Hour"],
                        how="left")
complete_campd_mt.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,gross_load_lag1,hourly_ramping_factor
0,MT,Glendive Generating Station,2176,GT-2,2019-01-01,0,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,NaN,NaN
1,MT,Glendive Generating Station,2176,GT-2,2019-01-01,1,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
2,MT,Glendive Generating Station,2176,GT-2,2019-01-01,2,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
3,MT,Glendive Generating Station,2176,GT-2,2019-01-01,3,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
4,MT,Glendive Generating Station,2176,GT-2,2019-01-01,4,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
5,MT,Glendive Generating Station,2176,GT-2,2019-01-01,5,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
6,MT,Glendive Generating Station,2176,GT-2,2019-01-01,6,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
7,MT,Glendive Generating Station,2176,GT-2,2019-01-01,7,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
8,MT,Glendive Generating Station,2176,GT-2,2019-01-01,8,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0
9,MT,Glendive Generating Station,2176,GT-2,2019-01-01,9,0.0,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0,0.0


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,MT,Glendive Generating Station,2176,GT-2,2019-01-01,0,0.0,0.0,NaN,NaN,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,NaN
1,MT,Glendive Generating Station,2176,GT-2,2019-01-01,1,0.0,0.0,NaN,NaN,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0
2,MT,Glendive Generating Station,2176,GT-2,2019-01-01,2,0.0,0.0,NaN,NaN,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0
3,MT,Glendive Generating Station,2176,GT-2,2019-01-01,3,0.0,0.0,NaN,NaN,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0
4,MT,Glendive Generating Station,2176,GT-2,2019-01-01,4,0.0,0.0,NaN,NaN,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,NaN,0.0


In [40]:
complete_campd_mt.to_csv('MT/Hourly/Concatted File/mt_2019_2024_complete.csv', index=False)

<h2>Transform and merge EIA-860 Data using crosswalk</h2>

In [41]:
# read eia generator data
eia_generator = pd.read_csv('EIA/Concatted File/eia_generator_2019_2023_MT.csv')

# these cols have blank strings (" ") that should be nan
columns_to_clean = [
    "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)",
    "Minimum Load (MW)", "Planned Retirement Year"
]

# replace blank strings with nan (any strings with only whitespace)
eia_generator[columns_to_clean] = eia_generator[columns_to_clean].replace(r'^\s*$', np.nan, regex=True)

# now define full dtype dictionary
eia_generator_type_dict = {
    "Year": "Int64", 
    "Utility ID": "string",
    "Utility Name": "string",
    "Plant Code": "Int64",  
    "Plant Name": "string",
    "State": "string",
    "Generator ID": "string",
    "Technology": "string",
    "Prime Mover": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Power Factor": "float64", 
    "Summer Capacity (MW)": "float64",  
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Operating Year": "Int64",
    "Planned Retirement Year": "float64"
}

# filter needed cols
eia_generator = eia_generator[["Year","Utility ID","Utility Name", "Plant Code", "Plant Name", "State", "Generator ID", "Technology", "Prime Mover", "Nameplate Capacity (MW)", "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)", "Minimum Load (MW)", "Operating Year", "Planned Retirement Year"]]

# Artificially extend the unit_level dataframe to include 2024 data (just copy the 2023 data and set Year to 2024)
eia_generator_2023 = eia_generator[eia_generator["Year"] == 2023].copy()
eia_generator_2024 = eia_generator_2023.copy()
eia_generator_2024["Year"] = 2024
eia_generator = pd.concat([eia_generator, eia_generator_2024], ignore_index=True)

# cast all dtypes according to the dictionary
eia_generator = eia_generator.astype(eia_generator_type_dict)
eia_generator


,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,2019,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT-2,Natural Gas Fired Combustion Turbine,GT,43.0,0.85,40.1,43.2,10.0,2003,NaN
1,2019,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT1,Natural Gas Fired Combustion Turbine,GT,40.7,0.90,34.8,40.9,9.0,1979,NaN
2,2019,15298,Talen Montana LLC,6076,Colstrip,MT,1,Conventional Steam Coal,ST,358.0,0.95,307.0,307.0,90.0,1975,2020.0
3,2019,15298,Talen Montana LLC,6076,Colstrip,MT,2,Conventional Steam Coal,ST,358.0,0.95,307.0,307.0,90.0,1976,2020.0
4,2019,15298,Talen Montana LLC,6076,Colstrip,MT,3,Conventional Steam Coal,ST,823.7,0.95,740.0,740.0,200.0,1984,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,2024,16233,Heorot Power Management,55749,Hardin Generator Project,MT,UNT1,Conventional Steam Coal,ST,115.7,0.85,107.0,107.0,50.0,2006,NaN
66,2024,1307,Basin Electric Power Coop,56606,Culbertson Generation Station,MT,01,Natural Gas Fired Combustion Turbine,GT,108.2,0.85,85.0,95.0,30.0,2010,NaN
67,2024,56815,NorthWestern Energy (MT wind/thermal),56908,Dave Gates Generating Station,MT,1,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN
68,2024,56815,NorthWestern Energy (MT wind/thermal),56908,Dave Gates Generating Station,MT,2,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN


In [42]:
crosswalk = pd.read_csv("epa_eia_crosswalk.csv")
crosswalk = crosswalk[(crosswalk["CAMD_STATE"] == "NY") | (crosswalk["CAMD_STATE"] == "MT") | (crosswalk["CAMD_STATE"] == "WY")] # filter for the states we need
crosswalk[crosswalk["CAMD_PLANT_ID"] == 2490]

# the col "CAMD_STATUS_DATE" gives the date where changes where last made for this plant id and unit id combo. we have to create new rows for every individual year up until the next change and so on with an additional col "Year"
# to match with the campd data
# join on CAMP_PLANT_ID, CAMD_UNIT_ID and add CAMD_NAMEPLATE_CAPACITY, CAMD_STATUS_DATE, EIA_NAMEPLATE_CAPACITY, EIA_BOILER_ID

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,...,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER
1772,1773,NY,Arthur Kill,2490,20,2,376.2,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,20,2,2,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1773,1774,NY,Arthur Kill,2490,30,3,535.5,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,30,3,3,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1774,1775,NY,Arthur Kill,2490,CT0001,GT1,16.3,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,NaN,NaN,GT1,3_1_Generator (generators) match on plant and ...,NaN


In [43]:
# create yearly rows manually

crosswalk["CAMD_STATUS_DATE"] = pd.to_datetime(crosswalk["CAMD_STATUS_DATE"])

crosswalk_sorted = crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "CAMD_STATUS_DATE"])

# Define end year up to which we generate rows
default_emt_year = 2024

expanded_rows = [] # save rows for expansion

# Group by plant/unit and generate yearly rows
for (plant_id, unit_id), group in crosswalk_sorted.groupby(["CAMD_PLANT_ID", "CAMD_UNIT_ID"]):
    group = group.sort_values("CAMD_STATUS_DATE").reset_index(drop=True)

    for i in range(len(group)):
        start_year = group.loc[i, "CAMD_STATUS_DATE"].year
        emt_year = (group.loc[i + 1, "CAMD_STATUS_DATE"].year 
                    if i + 1 < len(group) 
                    else default_emt_year + 1)  # +1 so we include the final year

        for year in range(start_year, emt_year):
            row = group.loc[i].copy()
            row["Year"] = year
            expanded_rows.append(row)

# Combine into new df
expanded_crosswalk = pd.DataFrame(expanded_rows)

expanded_crosswalk = expanded_crosswalk[expanded_crosswalk["Year"] >= 2019]
pd.set_option('display.max_columns', None)
expanded_crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "Year"])

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year
0,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176.0,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2019
0,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176.0,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2020
0,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176.0,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2021
0,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176.0,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2022
0,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176.0,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2020
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2021
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2022
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2023


In [44]:
expanded_crosswalk["EIA_PLANT_ID"] = expanded_crosswalk["EIA_PLANT_ID"].astype("Int64")
expanded_crosswalk["EIA_GENERATOR_ID"] = expanded_crosswalk["EIA_GENERATOR_ID"].astype("string")
expanded_crosswalk["Year"] = expanded_crosswalk["Year"].astype("Int64")

In [45]:
# join crosswalk with additional eia_generator data
combined_crosswalk = expanded_crosswalk.merge(eia_generator, how="left", left_on=["EIA_PLANT_ID", "EIA_GENERATOR_ID", "Year"], right_on=["Plant Code", "Generator ID", "Year"])
combined_crosswalk

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2019,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT-2,Natural Gas Fired Combustion Turbine,GT,43.0,0.85,40.1,43.2,10.0,2003,NaN
1,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2020,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT-2,Natural Gas Fired Combustion Turbine,GT,43.0,0.85,40.1,43.2,10.0,2003,NaN
2,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2021,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT-2,Natural Gas Fired Combustion Turbine,GT,43.0,0.85,40.1,43.2,10.0,2003,NaN
3,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2022,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT-2,Natural Gas Fired Combustion Turbine,GT,43.0,0.85,40.1,43.2,10.0,2003,NaN
4,1532,MT,Glendive Generating Station,2176,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.74,OPR,2003-06-01,0,GT-2,GT-2,MT,Glendive GT,2176,GT-2,43.0,NaN,GT,NG,47.0539,-104.74,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2023,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT-2,Natural Gas Fired Combustion Turbine,GT,43.0,0.85,40.1,43.2,10.0,2003,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2329,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2020,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2330,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2021,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2331,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2022,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2332,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.75,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,N

In [46]:
# read in cleaned dfs
campd_mt = pd.read_csv("MT/Hourly/Concatted File/mt_2019_2024_complete.csv", dtype=campd_type_dict)

In [47]:
combined_crosswalk = expanded_crosswalk.merge(eia_generator, how="right", left_on=["EIA_PLANT_ID", "EIA_GENERATOR_ID", "Year"], right_on=["Plant Code", "Generator ID", "Year"])
combined_crosswalk

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,1532.0,MT,Glendive Generating Station,2176.0,GT-2,GT-2,43.0,Pipeline Natural Gas,47.0539,-104.7400,OPR,2003-06-01,0.0,GT-2,GT-2,MT,Glendive GT,2176,GT-2,43.0,NaN,GT,NG,47.053900,-104.740000,0.0,0.0,2176.0,NaN,NaN,GT-2,3_1_Generator (generators) match on plant and ...,NaN,2019,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT-2,Natural Gas Fired Combustion Turbine,GT,43.0,0.85,40.1,43.2,10.0,2003,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,12199,Montana-Dakota Utilities Co,2176,Glendive GT,MT,GT1,Natural Gas Fired Combustion Turbine,GT,40.7,0.90,34.8,40.9,9.0,1979,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,15298,Talen Montana LLC,6076,Colstrip,MT,1,Conventional Steam Coal,ST,358.0,0.95,307.0,307.0,90.0,1975,2020.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,15298,Talen Montana LLC,6076,Colstrip,MT,2,Conventional Steam Coal,ST,358.0,0.95,307.0,307.0,90.0,1976,2020.0
4,3000.0,MT,Colstrip,6076.0,3,3,778.0,Coal,45.8831,-106.6140,OPR,1984-01-10,0.0,3,3,MT,Colstrip,6076,3,823.7,3,ST,SUB,45.883100,-106.614000,0.0,0.0,6076.0,3,3,3,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,15298,Talen Montana LLC,6076,Colstrip,MT,3,Conventional Steam Coal,ST,823.7,0.95,740.0,740.0,200.0,1984,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,6556.0,MT,Dave Gates Generating Station,56908.0,1B,1,67.7,Pipeline Natural Gas,46.1066,-112.8755,OPR,2010-08-31,0.0,1B,1,MT,Dave Gates Generating Station,56908,1,67.7,NaN,GT,NG,46.104713,-112.876605,0.0,0.0,56908.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2024,56815,NorthWestern Energy (MT wind/thermal),56908,Dave Gates Generating Station,MT,1,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN
84,6557.0,MT,Dave Gates Generating Station,56908.0,2A,2,67.7,Pipeline Natural Gas,46.1066,-112.8755,OPR,2010-09-07,0.0,2A,2,MT,Dave Gates Generating Station,56908,2,67.7,NaN,GT,NG,46.104713,-112.876605,0.0,0.0,56908.0,NaN,NaN,2,3_1_Generator (generators) match on plant and ...,NaN,2024,56815,NorthWestern Energy (MT wind/thermal),56908,Dave Gates Generating Station,MT,2,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN
85,6558.0,MT,Dave Gates Generating Station,56908.0,2B,2,67.7,Pipeline Natural Gas,46.1066,-112.8755,OPR,2010-09-07,0.0,2B,2,MT,Dave Gates Generating Station,56908,2,67.7,NaN,GT,NG,46.104713,-112.876605,0.0,0.0,56908.0,NaN,NaN,2,3_1_Generator (generators) match on plant and ...,NaN,2024,56815,NorthWestern Energy (MT wind/thermal),56908,Dave Gates Generating Station,MT,2,Natural Gas Fired Combustion Turbine,GT,67.7,0.85,44.1,48.1,5.0,2011,NaN
86,6559.0,MT,Dave Gates Generating Station,56908.0,3A,3,67.7,Pipeline Natural Gas,46.1066,-112.8755,OPR,2010-10-04,0

In [48]:
combined_crosswalk = combined_crosswalk[['CAMD_PLANT_ID',
       'CAMD_UNIT_ID', 'CAMD_GENERATOR_ID', 'CAMD_NAMEPLATE_CAPACITY', 'CAMD_STATUS_DATE', 'CAMD_RETIRE_YEAR', 'EIA_PLANT_ID',
       'EIA_GENERATOR_ID', 'EIA_NAMEPLATE_CAPACITY', 'EIA_BOILER_ID', 'EIA_RETIRE_YEAR', 'Year', 'Utility ID', 'Utility Name', 'Plant Code', 'Plant Name',
       'State', 'Generator ID', 'Technology', 'Prime Mover',
       'Nameplate Capacity (MW)', 'Nameplate Power Factor',
       'Summer Capacity (MW)', 'Winter Capacity (MW)', 'Minimum Load (MW)',
       'Operating Year', 'Planned Retirement Year']]

In [49]:
# bring to CAMD unit-level

agg_funcs = {
    'Nameplate Capacity (MW)': 'sum',
    'Summer Capacity (MW)': 'sum',
    'Winter Capacity (MW)': 'sum',
    'Minimum Load (MW)': 'sum',
    'Operating Year': ['min', 'max', 'mean'], # get operating year of the oldest (min) and newest (max) generator in the unit as well as the average across all gens
    'Planned Retirement Year': 'max',
}

unit_level = combined_crosswalk.groupby(['Year', 'EIA_PLANT_ID', 'CAMD_UNIT_ID']).agg(agg_funcs).reset_index()

display(unit_level)

# flatten multi-index columns (only keep the aggregation operator (min, max, mean) for Operating Year)
unit_level.columns = [
    f"{col[0]}_{col[1]}" if col[0] == "Operating Year" else col[0]
    for col in unit_level.columns.to_flat_index()
]

# set data types again for the final unit_level dataframe
unit_level = unit_level.astype({
    'Year': 'Int64',
    'EIA_PLANT_ID': 'Int64',
    'CAMD_UNIT_ID': 'string',
    'Nameplate Capacity (MW)': 'float64',
    'Summer Capacity (MW)': 'float64',
    'Winter Capacity (MW)': 'float64',
    'Minimum Load (MW)': 'float64',
    'Operating Year_min': 'Int64',
    'Operating Year_max': 'Int64',
    'Operating Year_mean': 'Int64',
    'Planned Retirement Year': 'Int64'
})

unit_level

Year EIA_PLANT_ID CAMD_UNIT_ID Nameplate Capacity (MW)  \
                                                       sum   
0   2019         2176         GT-2                    43.0   
1   2019         6076            3                   823.7   
2   2019         6076            4                   823.7   
3   2019        55749           U1                   115.7   
4   2019        56606         CT01                    91.0   
..   ...          ...          ...                     ...   
61  2024        56908           1B                    67.7   
62  2024        56908           2A                    67.7   
63  2024        56908           2B                    67.7   
64  2024        56908           3A                    67.7   
65  2024        56908           3B                    67.7   

   Summer Capacity (MW) Winter Capacity (MW) Minimum Load (MW) Operating Year  \
                    sum                  sum               sum            min   
0                  40.1                 43.2              10.0           2003   
1                 740.0                740.0             200.0           1984   
2                 740.0                740.0             200.0           1986   
3                 107.0                107.0              70.0           2006   
4                  85.0                 95.0              46.1           2010   
..                  ...                  ...               ...            ...   
61                 44.1                 48.1               5.0           2011   
62                 44.1                 48.1               5.0           2011   
63                 44.1                 48.1               5.0           2011   
64                 44.1                 48.1               5.0           2011   
65                 44.1                 48.1               5.0           2011   

                 Planned Retirement Year  
     max    mean                     max  
0   2003  2003.0                     NaN  
1   1984  1984.0                     NaN  
2   1986  1986.0                     NaN  
3   2006  2006.0                     NaN  
4   2010  2010.0                     NaN  
..   ...     ...                     ...  
61  2011  2011.0                     NaN  
62  2011  2011.0                     NaN  
63  2011  2011.0                     NaN  
64  2011  2011.0                     NaN  
65  2011  2011.0                     NaN  

[66 rows x 11 columns]

,Year,EIA_PLANT_ID,CAMD_UNIT_ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year
0,2019,2176,GT-2,43.0,40.1,43.2,10.0,2003,2003,2003,<NA>
1,2019,6076,3,823.7,740.0,740.0,200.0,1984,1984,1984,<NA>
2,2019,6076,4,823.7,740.0,740.0,200.0,1986,1986,1986,<NA>
3,2019,55749,U1,115.7,107.0,107.0,70.0,2006,2006,2006,<NA>
4,2019,56606,CT01,91.0,85.0,95.0,46.1,2010,2010,2010,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
61,2024,56908,1B,67.7,44.1,48.1,5.0,2011,2011,2011,<NA>
62,2024,56908,2A,67.7,44.1,48.1,5.0,2011,2011,2011,<NA>
63,2024,56908,2B,67.7,44.1,48.1,5.0,2011,2011,2011,<NA>
64,2024,56908,3A,67.7,44.1,48.1,5.0,2011,2011,2011,<NA>


In [50]:
# Calc min, max, and mean ages of the generators in the unit
unit_level['unit_age_min'] = unit_level["Year"] - unit_level['Operating Year_min']
unit_level['unit_age_max'] = unit_level["Year"] - unit_level['Operating Year_max']
unit_level['unit_age_mean'] = unit_level["Year"] - unit_level['Operating Year_mean']

In [51]:
unit_level = unit_level.rename(columns={
    "EIA_PLANT_ID": "Facility ID",
    "CAMD_UNIT_ID": "Unit ID"
})

In [52]:
# Merge each with unit_level
complete_campd_mt = pd.merge(campd_mt, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')

In [53]:
# Calculate differences between aggregated EIA Nameplate Capacity and CAMD Nameplate Capacity

# Create a column for absolute difference
complete_campd_mt["Capacity Difference"] = (
    complete_campd_mt["Nameplate Capacity (MWe)"] - complete_campd_mt["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_mt.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_mt = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference MT: {average_difference_mt:.2f} MW")

Average Capacity Difference MT: 9.56 MW


In [54]:
# fill EIA columns with na if it capacity difference exceeds a certain threshold

# rename the EIA column to make it clearer
for df in [complete_campd_mt]:
    df.rename(columns={
        "Nameplate Capacity (MW)": "Nameplate Capacity (MW) - EIA"
    }, inplace=True)

threshold = 30 
columns_to_null = [
    "Nameplate Capacity (MW) - EIA", 
    "Summer Capacity (MW)_y",
    "Winter Capacity (MW)_y",
    "Minimum Load (MW)_y"
]

# Recalculate capacity difference
complete_campd_mt["Capacity Difference"] = (
    complete_campd_mt["Nameplate Capacity (MWe)"] - complete_campd_mt["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_mt["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_mt.loc[condition, columns_to_null] = np.nan

In [55]:
# Calculate capacity difference again for comparison

# Create a column for absolute difference
complete_campd_mt["Capacity Difference"] = (
    complete_campd_mt["Nameplate Capacity (MWe)"] - complete_campd_mt["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_mt.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_mt = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference MT: {average_difference_mt:.2f} MW")

Average Capacity Difference MT: 1.53 MW


In [56]:
# Rename crosswalk cols and remove manual EIA transformation cols

# Define the renaming mapping
rename_cols = {
    'Summer Capacity (MW)_y': 'Summer Capacity (MW)',
    'Winter Capacity (MW)_y': 'Winter Capacity (MW)',
    'Minimum Load (MW)_y': 'Minimum Load (MW)',
    'Operating Year_min': 'Operating Year_min',
    'Operating Year_max': 'Operating Year_max',
    'Operating Year_mean': 'Operating Year_mean',
    'Planned Retirement Year_y': 'Planned Retirement Year',
    'unit_age_min_y': 'unit_age_min',
    'unit_age_max_y': 'unit_age_max',
    'unit_age_mean_y': 'unit_age_mean',
}


In [57]:
# Clean MT dataset
complete_campd_mt = complete_campd_mt.drop(columns=[
    'Summer Capacity (MW)_x', 'Winter Capacity (MW)_x',
    'Minimum Load (MW)_x', 'Planned Retirement Year_x',
    'unit_age_min_x', 'unit_age_max_x', 'unit_age_mean_x',
    'Capacity Difference', 'Nameplate Capacity (MW) - EIA'
])
complete_campd_mt = complete_campd_mt.rename(columns=rename_cols)

In [58]:
complete_campd_mt.to_csv('MT/Hourly/Concatted File/mt_2019_2024_complete_cleaned.csv', index=False)

In [59]:
complete_campd_mt

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),SO2 Mass Measure Indicator,SO2 Rate (lbs/mmBtu),SO2 Rate Measure Indicator,CO2 Mass (short tons),CO2 Mass Measure Indicator,CO2 Rate (short tons/mmBtu),CO2 Rate Measure Indicator,NOx Mass (lbs),NOx Mass Measure Indicator,NOx Rate (lbs/mmBtu),NOx Rate Measure Indicator,Heat Input (mmBtu),Heat Input Measure Indicator,Primary Fuel Type,Secondary Fuel Type,Unit Type,SO2 Controls,NOx Controls,Program Code,Year,Source Category,Latitude,Longitude,SO2 Phase,NOx Phase,Operating Status,Max Hourly HI Rate (mmBtu/hr),Associated Generators & Nameplate Capacity (MWe),Nameplate Capacity (MWe),Month,Day,hourly_capacity_factor,hourly_ramping_factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean
0,MT,Glendive Generating Station,2176,GT-2,2019-01-01,0,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Combustion turbine,Nothing,Dry Low NOx Burners,ARP,2019,Electric Utility,47.0539,-104.7400,Phase 2,Nothing,Operating,420.8,GT-2 (43),43.0,1,1,NaN,NaN,40.1,43.2,10.0,2003,2003,2003,<NA>,16,16,16
1,MT,Glendive Generating Station,2176,GT-2,2019-01-01,1,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Combustion turbine,Nothing,Dry Low NOx Burners,ARP,2019,Electric Utility,47.0539,-104.7400,Phase 2,Nothing,Operating,420.8,GT-2 (43),43.0,1,1,NaN,0.000000,40.1,43.2,10.0,2003,2003,2003,<NA>,16,16,16
2,MT,Glendive Generating Station,2176,GT-2,2019-01-01,2,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Combustion turbine,Nothing,Dry Low NOx Burners,ARP,2019,Electric Utility,47.0539,-104.7400,Phase 2,Nothing,Operating,420.8,GT-2 (43),43.0,1,1,NaN,0.000000,40.1,43.2,10.0,2003,2003,2003,<NA>,16,16,16
3,MT,Glendive Generating Station,2176,GT-2,2019-01-01,3,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Combustion turbine,Nothing,Dry Low NOx Burners,ARP,2019,Electric Utility,47.0539,-104.7400,Phase 2,Nothing,Operating,420.8,GT-2 (43),43.0,1,1,NaN,0.000000,40.1,43.2,10.0,2003,2003,2003,<NA>,16,16,16
4,MT,Glendive Generating Station,2176,GT-2,2019-01-01,4,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Combustion turbine,Nothing,Dry Low NOx Burners,ARP,2019,Electric Utility,47.0539,-104.7400,Phase 2,Nothing,Operating,420.8,GT-2 (43),43.0,1,1,NaN,0.000000,40.1,43.2,10.0,2003,2003,2003,<NA>,16,16,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620275,MT,Dave Gates Generating Station,56908,3B,2024-12-31,19,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Combustion turbine,Nothing,Water Injection|Selective Catalytic Reduction,ARP,2024,Electric Utility,46.1066,-112.8755,Phase 2,Nothing,Operating,296.4,3 (67.7),67.7,12,31,NaN,-0.029542,44.1,48.1,5.0,2011,2011,2011,<NA>,13,13,13
620276,MT,Dave Gates Generating Station,56908,3B,2024-12-31,20,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Combustion turbine,Nothing,Water Injection|Selective Catalytic Reduction,ARP,2024,Electric Utility,46.1066,-112.8755,Phase 2,Nothing,Operating,296.4,3 (67.7),67.7,12,31,NaN,0.000000,44.1,48.1,5.0,2011,2011,2011,<NA>,13,13,13
620277,MT,Dave Gates Generating Station,56908,3B,2024-12-31,21,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Diesel Oil,Com